# aipywidgets basic form

This notebook exercises the current non-AI prototype: fields, structured values, hooks, and actions.

In [1]:
from aipywidgets import AIForm, Action, fields

## Minimal form with an action

Edit the fields and press **Save**. The action handler prints the current form values below the form.

In [2]:
paper_form = AIForm(
    title="Paper metadata",
    steps=[
        {
            "id": "main",
            "label": "Main",
            "fields": [
                fields.Text("doi", label="DOI", full_width=True),
                fields.Text("title", label="Title", full_width=True),
                fields.Int("year", label="Year"),
            ],
        }
    ],
    actions=[
        Action(id="save", label="Save", style="primary"),
    ],
)

@paper_form.on_action("save")
def save(ctx):
    ctx.info(f"Saved: {ctx.values}")

paper_form

## Hook demo

Changing **Title** updates **Slug** through a Python hook.

In [3]:
hook_form = AIForm(
    title="Hook demo",
    steps=[
        {
            "id": "main",
            "label": "Main",
            "fields": [
                fields.Text("title", label="Title", full_width=True),
                fields.Text("slug", label="Slug", full_width=True),
            ],
        }
    ],
    actions=[Action(id="save", label="Save")],
)

@hook_form.on_change("title")
def update_slug(ctx):
    ctx.set_value("slug", ctx.value.lower().replace(" ", "-"))

@hook_form.on_action("save")
def save_hook(ctx):
    ctx.info(f"Saved: {ctx.values}")

hook_form

## Structured values

`Array(Object(...))` values can be read and updated through field paths.

In [4]:
authors_form = AIForm(
    title="Authors",
    steps=[
        {
            "id": "main",
            "label": "Main",
            "fields": [
                fields.Array(
                    "authors",
                    label="Authors",
                    item=fields.Object(
                        fields=[
                            fields.Text("given_name", label="Given name"),
                            fields.Text("family_name", label="Family name"),
                            fields.Text("orcid", label="ORCID"),
                        ],
                    ),
                    default=[
                        {
                            "given_name": "Ada",
                            "family_name": "Lovelace",
                            "orcid": "",
                        }
                    ],
                )
            ],
        }
    ],
    actions=[Action(id="save", label="Save")],
)

@authors_form.on_action("save")
def save_authors(ctx):
    ctx.info(f"Saved: {ctx.values}")

authors_form

In [5]:
authors_form.get_values()

{'authors': [{'given_name': 'Ada', 'family_name': 'Lovelace', 'orcid': ''}]}

In [6]:
authors_form.set_value("authors[0].family_name", "Byron")
authors_form.get_value("authors[0].family_name")

'Byron'

## Wizard demo

The wizard shows one step at a time. **Next** validates the current step, and the final step replaces **Next** with **Save** in the same navigation row.

In [7]:
wizard_form = AIForm(
    title="Dataset registration wizard",
    steps=[
        {
            "id": "file",
            "label": "File",
            "fields": [
                fields.Text("source_name", label="Source name", required=True, full_width=True),
                fields.Text("checksum", label="Checksum", full_width=True),
            ],
        },
        {
            "id": "metadata",
            "label": "Metadata",
            "fields": [
                fields.Text("title", label="Title", required=True, full_width=True),
                fields.Textarea("description", label="Description", full_width=True),
            ],
        },
        {
            "id": "review",
            "label": "Review",
            "fields": [
                fields.Checkbox("confirmed", label="Confirmed", required=True),
            ],
        },
    ],
    actions=[Action(id="save", label="Save", style="primary")],
)

@wizard_form.on_action("save")
def save_wizard(ctx):
    ctx.info(f"Saved wizard values: {ctx.values}")

wizard_form

## Local file selection

Run the next cell and enter a local root path when prompted. Folder selections are returned with a trailing `/`.

In [ ]:
import os

root_path = input("Local file selector root path: ").strip()
if not root_path:
    raise RuntimeError("Root path is required")

file_form = AIForm(
    title="Local file selection",
    steps=[
        {
            "id": "files",
            "label": "Files",
            "fields": [
                fields.Headline("heading", content="Choose files to export"),
                fields.Expression(
                    "description",
                    content="Files are returned as paths relative to the configured root directory.",
                ),
                fields.LocalFileSelect(
                    "selected_files",
                    label="Available files",
                    root_path=root_path,
                    full_width=True,
                ),
            ],
        }
    ],
    actions=[Action(id="save", label="Save", style="primary")],
)

@file_form.on_action("save")
def save_files(ctx):
    ctx.info(f"Selected paths: {ctx.get_value('selected_files')}")

file_form